In [ ]:
import os
import json
import pickle
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager
from IPython.display import display, HTML
from wordcloud import WordCloud

import torch

In [ ]:
# --- Model config ---
config_path = '../models/best_results/best_mean_trf_config.json'
variant     = 'best_auc'   # 'best_f1' / 'best_auc' / 'best_loss'

In [ ]:
plt.style.use('ggplot')

mpl.rcParams.update({
    'font.family':      'Ubuntu Sans',
    'axes.titlesize':   14,
    'axes.labelsize':   12,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
    'legend.fontsize':  10,
    'text.color':       'black',
    'axes.titlecolor':  'black',
    'axes.labelcolor':  'black',
    'xtick.color':      'black',
    'ytick.color':      'black',
    'figure.dpi':       120,
    'figure.facecolor': 'white',
})

font_path = font_manager.findfont("DejaVu Serif")

TOP_N        = 100
DISPLAY_N    = 10
EPS          = 1e-9
MIN_DOC_FREQ = 0.30

WC_KWARGS = dict(
    width=900, height=600, background_color='white',
    max_words=TOP_N, collocations=False,
    color_func=lambda *args, **kwargs: "black",
    font_path=font_path,
)

## Load Data

In [ ]:
processed_dir = os.path.join('..', 'data', 'processed')

with open(os.path.join(processed_dir, 'words_test.pkl'), 'rb') as f:
    df_words_test = pickle.load(f)

print(f"Loaded {len(df_words_test)} test scripts")
df_words_test.head()

In [ ]:
with open(config_path, 'r') as f:
    cfg = json.load(f)

training_cfg      = cfg['training']
checkpoint_prefix = training_cfg.get('checkpoint_prefix', 'model')
results_dir       = Path(training_cfg.get('results_dir', '../results')) / training_cfg['sub_dir']
models_dir        = Path(training_cfg.get('models_dir', '../models'))   / training_cfg['sub_dir']

# Load threshold from checkpoint

ckpt = torch.load(models_dir / f"{checkpoint_prefix}_{variant}.pth",
                  map_location='cpu', weights_only=False)
threshold = float(ckpt['val_threshold']) if 'val_threshold' in ckpt else 0.5
print(f"Threshold: {threshold:.4f}")

# Load predictions
results_file = results_dir / f"{checkpoint_prefix}_{variant}_test_results.json"
with open(results_file, 'r') as f:
    results_df = pd.DataFrame(json.load(f))

results_df['pred'] = (results_df['model_prob'] >= threshold).astype(int)

print(f"Loaded {len(results_df)} predictions")
results_df.head()

In [ ]:
# Merge predictions onto word tokens by position (same ordering as test set)
assert len(results_df) == len(df_words_test), "Prediction / token count mismatch!"

df = df_words_test.copy().reset_index(drop=True)
df['pred']      = results_df['pred'].values
df['model_prob'] = results_df['model_prob'].values
df['target']    = results_df['target'].values

# Quadrant labels
def quadrant(row):
    if row['target'] == 1 and row['pred'] == 1: 
        return 'TP'
    if row['target'] == 0 and row['pred'] == 1: 
        return 'FP'
    if row['target'] == 1 and row['pred'] == 0: 
        return 'FN'
    return 'TN'

df['quadrant'] = df.apply(quadrant, axis=1)

print(df['quadrant'].value_counts().to_string())
print(f"\nModel predicted nominated: {df['pred'].sum()}  /  {len(df)}")

## Helper Functions

In [ ]:
def discriminative_tables(group_a, group_b, label_a='A', label_b='B',
                           eps=EPS, min_doc_freq=MIN_DOC_FREQ, top_n=TOP_N):
    """Returns top words discriminating group_a vs group_b and vice versa."""
    def relative_freqs(subset):
        counts = Counter(t for tokens in subset['tokens'] for t in tokens)
        total  = sum(counts.values())
        return {w: c / total for w, c in counts.items()}

    rf_a = relative_freqs(group_a)
    rf_b = relative_freqs(group_b)

    df_freq_a, df_freq_b = defaultdict(int), defaultdict(int)
    for tokens in group_a['tokens']:
        for w in set(tokens): 
            df_freq_a[w] += 1
    for tokens in group_b['tokens']:
        for w in set(tokens): 
            df_freq_b[w] += 1

    all_words = set(rf_a) | set(rf_b)
    rows = [
        {
            'word':      w,
            'rf_a':      rf_a.get(w, 0),
            'rf_b':      rf_b.get(w, 0),
            'ratio_a':   rf_a.get(w, eps) / rf_b.get(w, eps),
            'ratio_b':   rf_b.get(w, eps) / rf_a.get(w, eps),
            'df_a':      df_freq_a[w] / max(len(group_a), 1),
            'df_b':      df_freq_b[w] / max(len(group_b), 1),
        }
        for w in all_words
    ]
    df_words = pd.DataFrame(rows)

    top_a = (
        df_words[df_words['df_a'] >= min_doc_freq]
        .nlargest(top_n, 'ratio_a')[['word', 'rf_a', 'ratio_a']]
        .rename(columns={'rf_a': f'rel_freq_{label_a}', 'ratio_a': f'ratio_vs_{label_b}'})
        .reset_index(drop=True)
    )
    top_b = (
        df_words[df_words['df_b'] >= min_doc_freq]
        .nlargest(top_n, 'ratio_b')[['word', 'rf_b', 'ratio_b']]
        .rename(columns={'rf_b': f'rel_freq_{label_b}', 'ratio_b': f'ratio_vs_{label_a}'})
        .reset_index(drop=True)
    )
    return top_a, top_b


def side_by_side_html(top_a, top_b, label_a, label_b, n=DISPLAY_N):
    col_a  = f'rel_freq_{label_a}'
    col_b  = f'rel_freq_{label_b}'
    rat_a  = f'ratio_vs_{label_b}'
    rat_b  = f'ratio_vs_{label_a}'
    left_html  = top_a.head(n).style.format({col_a: '{:.6f}', rat_a: '{:.2f}'}).to_html()
    right_html = top_b.head(n).style.format({col_b: '{:.6f}', rat_b: '{:.2f}'}).to_html()
    return f"""
    <table style="width:100%"><tr>
        <td style="width:50%; vertical-align:top; padding-right:20px">
            <b>More frequent in {label_a}</b><br>{left_html}
        </td>
        <td style="width:50%; vertical-align:top">
            <b>More frequent in {label_b}</b><br>{right_html}
        </td>
    </tr></table>
    """


def word_cloud_pair(top_a, top_b, label_a, label_b, title):
    col_a = f'rel_freq_{label_a}'
    col_b = f'rel_freq_{label_b}'
    wc_a = WordCloud(**WC_KWARGS).generate_from_frequencies(
        dict(zip(top_a['word'], top_a[col_a]))
    )
    wc_b = WordCloud(**WC_KWARGS).generate_from_frequencies(
        dict(zip(top_b['word'], top_b[col_b]))
    )
    fig, axes = plt.subplots(1, 2, figsize=(20, 7))
    axes[0].imshow(wc_a, interpolation='bilinear')
    axes[0].axis('off')
    axes[0].set_title(f'More frequent in {label_a}', fontsize=13, pad=10)
    axes[1].imshow(wc_b, interpolation='bilinear')
    axes[1].axis('off')
    axes[1].set_title(f'More frequent in {label_b}', fontsize=13, pad=10)
    plt.suptitle(title, fontsize=15, y=1.01)
    plt.subplots_adjust(wspace=0.05)
    plt.show()

## Section 1 — Predicted Nominated vs Predicted Not Nominated

All scripts the model called nominated vs all it called not-nominated, regardless of ground truth.

In [ ]:
pred_pos = df[df['pred'] == 1]
pred_neg = df[df['pred'] == 0]

print(f"Predicted nominated: {len(pred_pos)}  |  Predicted not nominated: {len(pred_neg)}")

top_pos, top_neg = discriminative_tables(
    pred_pos, pred_neg,
    label_a='PRED_NOM', label_b='PRED_NOT'
)

display(HTML(side_by_side_html(top_pos, top_neg, 'PRED_NOM', 'PRED_NOT')))

In [ ]:
word_cloud_pair(top_pos, top_neg, 'PRED_NOM', 'PRED_NOT',
                f'Top {TOP_N} Discriminative Words — Predicted Nominated vs Not Nominated')

## Section 2 — Confusion Matrix Quadrants

Word differences between TP / FP / FN / TN groups.

In [ ]:
tp = df[df['quadrant'] == 'TP']
fp = df[df['quadrant'] == 'FP']
fn = df[df['quadrant'] == 'FN']
tn = df[df['quadrant'] == 'TN']

for label, grp in [('TP', tp), ('FP', fp), ('FN', fn), ('TN', tn)]:
    print(f"{label}: {len(grp)} scripts")

In [ ]:
# TP vs FN: among actually-nominated scripts, what words did the model pick up on vs miss?
if len(tp) > 1 and len(fn) > 1:
    top_tp, top_fn = discriminative_tables(tp, fn, label_a='TP', label_b='FN')
    display(HTML('<h4>TP vs FN — words in correctly-spotted nominated scripts vs missed ones</h4>'))
    display(HTML(side_by_side_html(top_tp, top_fn, 'TP', 'FN')))
    word_cloud_pair(top_tp, top_fn, 'TP', 'FN',
                    'TP vs FN — What words did the model catch vs miss in nominated scripts?')
else:
    print("Not enough TP or FN samples for discriminative analysis.")

In [ ]:
# FP vs TN: among non-nominated scripts, which ones fooled the model vs didn't?
if len(fp) > 1 and len(tn) > 1:
    top_fp, top_tn = discriminative_tables(fp, tn, label_a='FP', label_b='TN')
    display(HTML('<h4>FP vs TN — words in non-nominated scripts that fooled the model vs ones it correctly dismissed</h4>'))
    display(HTML(side_by_side_html(top_fp, top_tn, 'FP', 'TN')))
    word_cloud_pair(top_fp, top_tn, 'FP', 'TN',
                    'FP vs TN — What fooled the model among non-nominated scripts?')
else:
    print("Not enough FP or TN samples for discriminative analysis.")